In [2]:
import tweepy
import pandas as pd
import json
import snscrape.modules.twitter as sntwitter
import time

We are going to use the twitter api with the library tweepy to get the followers and the following 

In [2]:
api_key = "LEqYs6bOKjU7HuNEGiGlU2eSS"
api_secrets = "0WukopJL0Sg7t6QJenUOO0LqojPyTWkSl8oOepDRfui5ybW1Ce"
access_token ="1148685416423395331-Y5rueG3D35JXUVDCNi3y7wHTUL2nMt"
access_secret="ZiVeiScfLKoRxgOAAR3SyJsCeVKli9BPtSxSttjbmkqRQ"

auth = tweepy.OAuthHandler(api_key,api_secrets,access_token, access_secret)

api = tweepy.API(auth)

try:
    api.verify_credentials()
    print('Success')
except:
    print('Fail')
    

Success


In [3]:
elon= "elonmusk"

Getting the following

In [5]:
following=[]
for page in tweepy.Cursor(api.get_friends, screen_name=elon,count=200).pages(10):
    for user in page:
        name = f"{user.name} (@{user.screen_name})"
        following.append(name)

In [28]:
len(following)

151

Save the following in a file using the json library

In [32]:
with open("following_raw","w") as fp:
    json.dump(following,fp)

Get the followers and save them in a file

After several tries, it was noticed that the limit rate was exceeded after 3000  followers were retrieved and also that the api needs 15 minutes to expires the timeout.

In [23]:
followers=[]
counter=0
for page in tweepy.Cursor(api.get_followers, screen_name=elon,count=200).pages(120):
    for user in page:
        name = f"{user.name} (@{user.screen_name})"
        followers.append(name)
        counter+=1
        if counter == 2999:
            counter=0
            time.sleep(15*60)   #sleep for 15 minutes
        if len(followers)>15000:
            break

TooManyRequests: 429 Too Many Requests
88 - Rate limit exceeded

In [26]:
len(followers)

33015

In [19]:
a=followers

In [22]:
b=followers

In [25]:
followers= a+b+followers

In [31]:
len(followers)

33015

Trying with different number of pages and time sleep, it was possible to get 33k followers

In [27]:
with open("followers_raw","w") as fp:
    json.dump(followers,fp)

We are going to use a webscraper to get elon's tweet. The tweets start from the beginning the 2018 because I think elon's notoriety has been increasing in the last 4 years.

The data extracted will contain the text, the number of likes, the number of retweets, the number of replies and the user mentions

In [3]:
tweets=[]
query = "(from:elonmusk) until:2022-12-11 since:2018-01-01"


start = time.time()
for tw in sntwitter.TwitterSearchScraper(query).get_items():
    tweets.append([tw.date,tw.content,tw.likeCount,tw.retweetCount,tw.replyCount, tw.mentionedUsers])
end = time.time()
print(end - start)

385.06362438201904


The data will be stored in dataframe and exported in a csv file using pandas 

In [4]:
col = ["Date", "Tweet", "Likes","Retweet","Reply","Mention"]
df=pd.DataFrame(tweets,columns = col )
df

,Date,Tweet,Likes,Retweet,Reply,Mention
0,2022-12-10 23:22:58+00:00,@Twitter And many other features to come!,24783,1680,2583,[https://twitter.com/Twitter]
1,2022-12-10 20:01:11+00:00,https://t.co/80DdvpsNjM,647964,67179,20023,None
2,2022-12-10 19:56:59+00:00,Twitter is both a social media company and a c...,692416,95127,35890,None
3,2022-12-10 19:29:34+00:00,@elizableu Looks like Yoel is arguing in favor...,61853,15325,6071,[https://twitter.com/elizableu]
4,2022-12-10 19:21:31+00:00,@elizableu This explains a lot,91369,6914,1628,[https://twitter.com/elizableu]
...,...,...,...,...,...,...
15630,2018-01-07 03:00:48+00:00,@mhmtkcn Near 405,1150,28,47,[https://twitter.com/mhmtkcn]
15631,2018-01-05 00:30:15+00:00,https://t.co/3k71xzDIP1,26022,3489,609,None
15632,2018-01-05 00:30:00+00:00,Falcon Heavy goes vertical https://t.co/uG1k0W...,80594,13898,1974,None
15633,2018-01-03 08:22:31+00:00,Using a neural net to detect rain using camera...,16442,2169,655,None


In [ ]:
df.to_csv('elon.csv')

To handle task E of the assignment, the tweets of @dr_jpe have been collected

In [8]:
jpe_tweets=[]
query = "(from:dr_jpe) until:2022-12-11 since:2022-01-01"

for tw in sntwitter.TwitterSearchScraper(query).get_items():
    jpe_tweets.append([tw.date,tw.content,tw.likeCount,tw.retweetCount,tw.replyCount, tw.mentionedUsers])

In [9]:
col = ["Date", "Tweet", "Likes","Retweet","Reply","Mention"]
df_jpe=pd.DataFrame(jpe_tweets,columns = col )
df_jpe['Mention']=df_jpe['Mention'].fillna('No mention')
df_jpe

,Date,Tweet,Likes,Retweet,Reply,Mention
0,2022-12-10 08:07:07+00:00,"Our book chapter ""#ML for Metabolomic Pathway ...",8,0,0,"[https://twitter.com/UMmalta, https://twitter...."
1,2022-12-07 11:23:12+00:00,Christmas would not be the same without the #A...,2,1,0,No mention
2,2022-12-06 12:51:24+00:00,"@dwejjaq_enormi if you have to explain it, it'...",0,0,0,[https://twitter.com/dwejjaq_enormi]
3,2022-11-23 20:25:15+00:00,Come at @UMmalta to work on some #Bioinformati...,3,2,0,[https://twitter.com/UMmalta]
4,2022-11-18 12:44:41+00:00,I would say I am at least at lvl5\nhttps://t.c...,0,0,0,No mention
5,2022-11-15 10:36:09+00:00,#CPS3235 Assignment. @elonmusk is king.,1,0,1,[https://twitter.com/elonmusk]
6,2022-11-15 08:36:30+00:00,These are *read* emails. @airmalta why do you...,2,0,3,[https://twitter.com/AirMalta]
7,2022-11-15 04:57:06+00:00,@TheMarvel91 @Campanelli11 Ma anche un banalis...,0,0,0,"[https://twitter.com/TheMarvel91, https://twit..."
8,2022-10-14 19:37:26+00:00,@drmenguin I'm always game to some \LaTeX fun 😂,3,0,0,[https://twitter.com/drmenguin]
9,2022-10-14 11:04:06+00:00,Great to see @danvlla with his few-shot #Machi...,6,2,1,"[https://twitter.com/danvlla, https://twitter...."


In [10]:
df_jpe.to_csv('jpe.csv')